# Cuaderno D · Índice socio-ambiental, coberturas, 2030 y candidatos

**MICROCLIMA · NASA Space Apps 2026 Cochabamba · Enfoque 2**

Este cuaderno **funciona solo**: no necesita que el Cuaderno C siga abierto, porque lee los archivos que C generó.

**Sube a este Colab** (ícono de carpeta → subir):

| Archivo | De dónde sale |
|---|---|
| `distritos_indicadores_ambientales_2024.geojson` | `public/data/` |
| `equipamientos.geojson` | Cuaderno C |
| `areas_equipamiento.geojson` | Cuaderno C |
| `indicadores_gamc_distritos.csv` | Cuaderno C |
| `tendencias_distritos.json` | `public/data/` |
| Las líneas de transporte: `lineas.zip` **o** el `.rar` del portal | Portal GAMC |
| *(Opcional)* `lluvias_inundaciones.json` | Cuaderno F de Erika. Si lo subes, se agregan alertas de inundación |

**Cambio importante respecto a la guía:** la capa de población por manzanas del portal suma unos 416 mil habitantes y parece incompleta. Por eso la población de cada distrito (hoy y en 2030) sale de la grilla **GHSL** de Earth Engine, que cubre todo el municipio. El dato del portal se guarda solo como referencia.

## Celda D0 · Preparación y grilla de población

In [ ]:
!pip install -q rasterio earthengine-api
import io, os, glob, zipfile, subprocess, json, requests, numpy as np, pandas as pd, matplotlib.pyplot as plt
import geopandas as gpd, rasterio.features, rasterio.transform, ee
from rasterio.io import MemoryFile
from scipy.spatial import cKDTree
from pyproj import Transformer

ee.Authenticate(); ee.Initialize(project='microclimas-cochabamba')

R_SALUD, R_COLEGIO, R_PARADA = 1000, 1000, 300   # metros (supuestos editables)
region = ee.Geometry.Rectangle([-66.2529, -17.5054, -66.0844, -17.2716])

dist = gpd.read_file('/content/distritos_indicadores_ambientales_2024.geojson').to_crs(3857)
dist['codigo_distrito'] = dist['codigo_distrito'].astype(float).astype(int)

def ghsl(anio):
    img = ee.Image(f'JRC/GHSL/P2023A/GHS_POP/{anio}').select('population_count')
    url = img.getDownloadURL({'region': region, 'scale': 100, 'crs': 'EPSG:3857', 'format': 'GEO_TIFF'})
    with MemoryFile(requests.get(url, timeout=300).content) as mf, mf.open() as src:
        return src.read(1).astype(float).clip(min=0), src.transform

G = {a: ghsl(a) for a in [2025, 2030]}
forma, trp = G[2025][0].shape, G[2025][1]
fi, co = np.indices(forma)
xs, ys = map(np.array, rasterio.transform.xy(trp, fi.ravel(), co.ravel()))
distr_px = rasterio.features.rasterize(zip(dist.geometry, dist.codigo_distrito), out_shape=forma,
                                       transform=trp, fill=0, dtype='int32').ravel()
pob_px = {a: G[a][0].ravel() for a in G}

pob_ghsl = pd.DataFrame({'d': distr_px, 'p25': pob_px[2025], 'p30': pob_px[2030]}).query('d > 0').groupby('d').sum().round(0)
print('Población GHSL del municipio · 2025:', int(pob_ghsl['p25'].sum()), '· 2030:', int(pob_ghsl['p30'].sum()))
print(pob_ghsl)

## Celda D1 · Datos del Cuaderno C y paradas de transporte

Si subiste el `.rar`, el cuaderno lo descomprime solo. Los 3 archivos dañados del portal se saltan con un aviso.

In [ ]:
equip = gpd.read_file('/content/equipamientos.geojson').to_crs(3857)
salud = equip[equip['tipo'] == 'salud'].copy()
colegios = equip[equip['tipo'] == 'educacion'].copy()
print('Valores de nivel de salud:', salud['nivel'].unique())
s1 = salud[salud['nivel'].astype(str).str.contains('1', na=False)]   # primer nivel (incluye '1er. Nivel')
print(len(salud), 'establecimientos de salud ·', len(s1), 'de primer nivel ·', len(colegios), 'colegios')

eq = gpd.read_file('/content/areas_equipamiento.geojson').to_crs(3857)
print(len(eq), 'terrenos de equipamiento')

ind = pd.read_csv('/content/indicadores_gamc_distritos.csv')
ind['codigo_distrito'] = ind['codigo_distrito'].astype(float).astype(int)
ind = ind.set_index('codigo_distrito')

# Líneas de transporte: zip o rar
os.makedirs('/content/lineas', exist_ok=True)
if os.path.exists('/content/lineas.zip'):
    zipfile.ZipFile('/content/lineas.zip').extractall('/content/lineas')
else:
    rars = glob.glob('/content/*.rar')
    if rars:
        subprocess.run('apt-get -qq install -y unrar > /dev/null', shell=True)
        subprocess.run(f'unrar x -o+ -inul "{rars[0]}" /content/lineas/', shell=True)
partes = []
for f in glob.glob('/content/lineas/**/*.geojson', recursive=True):
    try:
        g = gpd.read_file(f)
    except Exception:
        print('Se salta (dañado):', os.path.basename(f)); continue
    g = g.set_crs(4326) if g.crs is None else g
    partes.append(g[g.geom_type == 'Point'].to_crs(3857)[['geometry']])
paradas = pd.concat(partes)
paradas['k'] = [(round(p.x / 10), round(p.y / 10)) for p in paradas.geometry]
paradas = gpd.GeoDataFrame(paradas.drop_duplicates('k'), crs=3857)
print(len(paradas), 'paradas distintas')

## Celda D2 · Coberturas por distrito, hoy y en 2030

In [ ]:
def dist_min(puntos):
    return cKDTree(np.c_[puntos.geometry.x, puntos.geometry.y]).query(np.c_[xs, ys])[0]
d_salud, d_col, d_par = dist_min(s1), dist_min(colegios), dist_min(paradas)

def cobertura(pobv, d, R):
    g = pd.DataFrame({'d': distr_px, 'p': pobv, 'c': pobv * (d <= R)}).query('d > 0').groupby('d').sum()
    return (100 * g['c'] / g['p']).round(1)

cov = pd.DataFrame({'cobertura_salud': cobertura(pob_px[2025], d_salud, R_SALUD),
                    'cobertura_colegio': cobertura(pob_px[2025], d_col, R_COLEGIO),
                    'cobertura_transporte': cobertura(pob_px[2025], d_par, R_PARADA),
                    'cobertura_salud_2030': cobertura(pob_px[2030], d_salud, R_SALUD)})
cov['personas'] = pob_ghsl['p25']
cov['personas_2030'] = pob_ghsl['p30']
cov['factor_2030'] = (pob_ghsl['p30'] / pob_ghsl['p25']).round(3)
print(cov)

## Celda D3 · Índice socio-ambiental

La celda elige sola las columnas de LST y NDVI 2024 del GeoJSON y te muestra cuáles usó. **Revisa que sean los promedios** (media); si no, cámbialas en `LST_COL` y `NDVI_COL`.

In [ ]:
amb = dist.set_index('codigo_distrito')
def elegir(clave):
    cols = [c for c in amb.columns if clave in c.lower()]
    pref = [c for c in cols if any(p in c.lower() for p in ['mean', 'media', 'prom'])]
    return (pref or cols)[0]
LST_COL, NDVI_COL = elegir('lst'), elegir('ndvi')
print('Columnas disponibles:', amb.columns.tolist())
print('Usando →', LST_COL, 'y', NDVI_COL)

tj = json.load(open('/content/tendencias_distritos.json', encoding='utf-8'))
tend = {int(float(d['id'])): d['tendencias']['ndvi'] for d in tj['distritos']}

T = ind.drop(columns=['personas', 'viviendas'], errors='ignore').join(cov)
T['personas_gamc'] = ind.get('personas')          # solo referencia (capa incompleta)
T['lst'], T['ndvi'] = amb[LST_COL].astype(float), amb[NDVI_COL].astype(float)
T['tend_ndvi'] = [tend[i]['sen_pendiente_anual'] if tend[i]['significativa'] else 0 for i in T.index]
s1d = s1.dropna(subset=['codigo_distrito'])
T['salud_1er'] = s1d.groupby(s1d['codigo_distrito'].astype(float).astype(int)).size().reindex(T.index).fillna(0)
T['verde_por_hab'] = T['area_verde_m2'] / T['personas']
T['hab_por_salud'] = T['personas'] / T['salud_1er'].replace(0, np.nan)
T['hab_por_colegio'] = T['personas'] / T['colegios'].replace(0, np.nan)

COMP = {   # nombre visible: (columna, dimensión, ¿más alto es mejor?)
    'Calor del suelo': ('lst', 'ambiental', False),
    'Vegetación': ('ndvi', 'ambiental', True),
    'Tendencia de vegetación': ('tend_ndvi', 'ambiental', True),
    'Área verde por habitante': ('verde_por_hab', 'ambiental', True),
    'Habitantes por centro de salud de 1er nivel': ('hab_por_salud', 'servicios', False),
    'Habitantes por unidad educativa': ('hab_por_colegio', 'servicios', False),
    'Población cerca del transporte': ('cobertura_transporte', 'servicios', True)}

def puntaje(s, alto_mejor):
    s = s.astype(float)
    z = (s - s.min()) / (s.max() - s.min()) * 100 if s.max() > s.min() else s * 0 + 50
    return (z if alto_mejor else 100 - z).fillna(0)     # sin centro de salud = peor puntaje

PT = pd.DataFrame({k: puntaje(T[c], a) for k, (c, _, a) in COMP.items()})
T['dim_ambiental'] = PT[[k for k, v in COMP.items() if v[1] == 'ambiental']].mean(axis=1).round(1)
T['dim_servicios'] = PT[[k for k, v in COMP.items() if v[1] == 'servicios']].mean(axis=1).round(1)
T['indice'] = (0.5 * T['dim_ambiental'] + 0.5 * T['dim_servicios']).clip(lower=1).round(0)
T['semaforo'] = pd.cut(T['indice'], [0, 33, 66, 100], labels=['rojo', 'amarillo', 'verde']).astype(str)
T['ranking'] = T['indice'].rank(ascending=False, method='min').astype(int)
print(T[['nombre', 'indice', 'semaforo', 'dim_ambiental', 'dim_servicios', 'personas']].sort_values('indice'))

# Índice ambiental histórico (vegetación), comparable entre años
anual = {int(float(d['id'])): pd.DataFrame(d['serie']).assign(anio=lambda x: x['fecha'].str[:4].astype(int)).groupby('anio')['ndvi'].mean()
         for d in tj['distritos']}
A = pd.DataFrame(anual)
A = ((A - A.min().min()) / (A.max().max() - A.min().min()) * 99 + 1).round(1)

## Celda D4 · Puntos críticos (alertas con su razón)

Si subiste `lluvias_inundaciones.json` (de Erika), se agregan alertas de inundación.

In [ ]:
prom = T.mean(numeric_only=True)
inund = {}
if os.path.exists('/content/lluvias_inundaciones.json'):
    li = json.load(open('/content/lluvias_inundaciones.json', encoding='utf-8'))
    inund = {int(float(d['id'])): d for d in li['distritos']}
    print('Alertas de inundación activadas')

def alertas(i, r):
    a, t = [], tend[i]
    if t['significativa'] and t['sen_pendiente_anual'] < 0:
        a.append({'tipo': 'ambiental', 'texto': f"Pierde vegetación de forma significativa: {t['sen_pendiente_anual'] * 25:+.3f} de NDVI en 25 años."})
    if r['indice'] < 34:
        a.append({'tipo': 'indice', 'texto': f"Índice socio-ambiental en rojo: {int(r['indice'])} de 100."})
    if pd.isna(r['hab_por_salud']) or r['hab_por_salud'] > prom['hab_por_salud']:
        txt = 'No tiene centros de salud de primer nivel.' if pd.isna(r['hab_por_salud']) else \
              f"{int(r['hab_por_salud']):,} habitantes por centro de primer nivel (promedio municipal: {int(prom['hab_por_salud']):,})."
        a.append({'tipo': 'salud', 'texto': txt.replace(',', '.')})
    if r['hab_por_colegio'] > prom['hab_por_colegio']:
        a.append({'tipo': 'educacion', 'texto': f"{int(r['hab_por_colegio']):,} habitantes por unidad educativa (promedio: {int(prom['hab_por_colegio']):,}).".replace(',', '.')})
    if r['cobertura_transporte'] < prom['cobertura_transporte']:
        a.append({'tipo': 'transporte', 'texto': f"Solo {r['cobertura_transporte']:.0f} % de la población vive a menos de {R_PARADA} m de una parada."})
    if r['factor_2030'] > prom['factor_2030'] and r['cobertura_salud_2030'] < prom['cobertura_salud_2030']:
        a.append({'tipo': 'futura', 'texto': f"Crecerá {100 * (r['factor_2030'] - 1):.0f} % al 2030 con cobertura de salud baja ({r['cobertura_salud_2030']:.0f} %)."})
    d = inund.get(i)
    if d and (d['susceptibilidad']['nivel'] == 'Alta' or d.get('personas_franja_rocha', 0) > 1000):
        a.append({'tipo': 'inundacion', 'texto': f"Susceptibilidad a inundación {d['susceptibilidad']['nivel'].lower()}; "
                                                 f"{d.get('personas_franja_rocha', 0):,} personas viven a menos de 200 m del río Rocha.".replace(',', '.')})
    return a

for i, r in T.iterrows():
    print(r['nombre'], '→', [x['tipo'] for x in alertas(i, r)])

## Celda D5 · Candidatos del simulador de equipamientos

Una ubicación solo es candidata si está en un **terreno de equipamiento municipal** y tiene alrededor al menos 2.000 personas **sin cobertura en 2030**: así no proponemos centros en zonas vacías.

In [ ]:
inv = Transformer.from_crs(3857, 4326, always_xy=True)
eqc = eq.copy(); eqc['geometry'] = eqc.centroid
eqc = gpd.sjoin(eqc, dist[['codigo_distrito', 'geometry']], predicate='within', how='inner').drop(columns='index_right')
arbol = cKDTree(np.c_[xs, ys])

def candidatos(d_actual, R, tipo, minimo=2000, top=30):
    sin_cob = pob_px[2030] * (d_actual > R)
    out = []
    for _, r in eqc.iterrows():
        nueva = float(sin_cob[arbol.query_ball_point([r.geometry.x, r.geometry.y], R)].sum())
        if nueva >= minimo:
            lon, lat = inv.transform(r.geometry.x, r.geometry.y)
            out.append({'tipo': tipo, 'lon': round(lon, 6), 'lat': round(lat, 6), 'distrito': int(r['codigo_distrito']),
                        'area_terreno_m2': int(r['area_m2']), 'poblacion_nueva_cubierta_2030': int(nueva)})
    return sorted(out, key=lambda c: -c['poblacion_nueva_cubierta_2030'])[:top]

cands = candidatos(d_salud, R_SALUD, 'salud') + candidatos(d_col, R_COLEGIO, 'educacion')
print(len(cands), 'candidatos')
print(pd.DataFrame(cands).groupby('tipo').head(5))

## Celda D6 · Exportar todo

Al terminar, descarga `indice_distritos.json`, `candidatos_equipamiento.json` y `poblacion_grilla.json` a `public/data/` del proyecto.

In [ ]:
salida = {'meta': {
    'pesos': {'ambiental': 0.5, 'servicios': 0.5},
    'componentes': {k: {'dimension': v[1], 'alto_es_mejor': v[2]} for k, v in COMP.items()},
    'radios_m': {'salud': R_SALUD, 'colegio': R_COLEGIO, 'parada': R_PARADA},
    'poblacion': 'Grilla GHSL 2025 y 2030 (JRC). La capa de población por manzanas del portal GAMC suma unos 416 mil habitantes y parece incompleta; se guarda solo como referencia (personas_gamc).',
    'semaforo': {'rojo': '1–33', 'amarillo': '34–66', 'verde': '67–100'},
    'fuentes': ['GAMC: centros de salud, unidades educativas, áreas verdes, áreas de equipamiento, transporte, población por manzanas (referencia)',
                'NASA MODIS: NDVI 2001–2025', 'Landsat 8/9 y Sentinel-2: LST y NDVI 2024', 'JRC GHSL: grilla de población 2025 y 2030'],
    'notas': ['Una tendencia de vegetación no significativa cuenta como cero.',
              'Las ubicaciones de salud y colegios se obtuvieron desde los servicios de mapa del portal GAMC.',
              'La cobertura escolar es aproximada: muchos estudiantes estudian lejos de su casa.']},
  'distritos': []}
num = lambda v: None if v is None or pd.isna(v) else round(float(v), 2)
for i, r in T.sort_index().iterrows():
    salida['distritos'].append({
        'id': str(int(i)), 'nombre': r['nombre'], 'indice': int(r['indice']), 'semaforo': r['semaforo'], 'ranking': int(r['ranking']),
        'dimension_ambiental': num(r['dim_ambiental']), 'dimension_servicios': num(r['dim_servicios']),
        'componentes': {k: {'valor': num(r[c]), 'puntaje': num(PT.loc[i, k])} for k, (c, _, _) in COMP.items()},
        'historico': [{'anio': int(a), 'indice_ambiental': num(A.loc[a, i])} for a in A.index],
        'indicadores': {c: num(r.get(c)) for c in ['personas', 'personas_gamc', 'salud_1er', 'colegios', 'area_verde_m2', 'verde_por_hab',
                        'cobertura_salud', 'cobertura_colegio', 'cobertura_transporte', 'personas_2030', 'factor_2030', 'cobertura_salud_2030']},
        'alertas': alertas(i, r)})
json.dump(salida, open('indice_distritos.json', 'w', encoding='utf-8'), ensure_ascii=False)
json.dump(cands, open('candidatos_equipamiento.json', 'w', encoding='utf-8'), ensure_ascii=False)

# Grilla liviana para que el simulador recalcule coberturas en la app
ok = (distr_px > 0) & (pob_px[2025] > 0)
lon, lat = inv.transform(xs[ok], ys[ok])
grilla = np.c_[np.round(lon, 5), np.round(lat, 5), np.round(pob_px[2025][ok]), np.round(pob_px[2030][ok]), distr_px[ok]].tolist()
json.dump({'columnas': ['lon', 'lat', 'pob_2025', 'pob_2030', 'distrito'], 'celdas': grilla}, open('poblacion_grilla.json', 'w'))
print('Listo:', len(salida['distritos']), 'distritos,', len(cands), 'candidatos,', len(grilla), 'celdas')
print('Tamaño de la grilla:', round(os.path.getsize('poblacion_grilla.json') / 1e6, 1), 'MB')